# Multi-agent report generation with human approval 
<br>

Research Agent → Writer Agent → Human Review → Final Report

# 1 .llm

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b")

# 2  state

In [ ]:
from typing import TypedDict

class ReportState(TypedDict):
    topic: str
    research: str
    draft: str
    feedback: str
    final_report: str
    revision_cnt : int

In [ ]:
def research_agent(state: ReportState):
    print("Research Agent started")

    response = llm.invoke(
        f"""
        You are a research specialist.
        Prepare useful key points about this topic:
        {state['topic']}

        Organize the points under suitable headings.
        """
    )

    return {"research": response.content}

In [ ]:
def writer_agent(state: ReportState):
    print("Writer Agent started")

    response = llm.invoke(
        f"""
        You are a professional report writer.

        Topic: {state['topic']}

        Research notes:
        {state['research']}

        Feedback on the previous draft, if any:
        {state['feedback']}

        Write a clear, structured report.
        Incorporate the feedback if provided.
        """
    )

    return {
        "draft": response.content,
        "feedback": ""
    }

In [ ]:
from langgraph.types import interrupt,Command
from typing import Literal

def human_review(state: ReportState)-> Command[Literal["finalize", "writer"]]:
    print("Human Review started")

    decision = interrupt({
        "message": "Review the report",
        "draft": state["draft"],
        "options": ["approve", "edit", "reject"]
    })

    if decision["action"] == "approve":
        return Command(goto="finalize")

    elif decision["action"] == "edit":
        return Command(
            update={"draft": decision["edited_text"]},
            goto="finalize"
        )

    elif decision["action"] == "reject":
        revision_cnt = state.get("revision_cnt",0) + 1
        print(revision_cnt)
        if not revision_cnt > 3:
            return Command(
                update={"feedback": decision["feedback"],"revision_cnt":revision_cnt},
                goto="writer"
            )
        return Command(goto="finalize")

    raise ValueError("Unknown review action")

In [ ]:
def finalize(state: ReportState):
    print("Finalizing report")

    return {"final_report": state["draft"]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(ReportState)

builder.add_node("researcher", research_agent)
builder.add_node("writer", writer_agent)
builder.add_node("human_review", human_review)
builder.add_node("finalize", finalize)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "human_review")
builder.add_edge("finalize", END)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
config = {
    "configurable": {
        "thread_id": "report-101"
    }
}

result = graph.invoke(
    {
        "topic": "Benefits of AI in healthcare",
        "research": "",
        "draft": "",
        "feedback": "",
        "final_report": ""
    },
    config=config
)

# The graph pauses at human_review
print(result["__interrupt__"])

In [ ]:
# result = graph.invoke(
#     Command(resume={"action": "approve"}),
#     config=config
# )

# print(result["final_report"])

In [ ]:
# result = graph.invoke(
#     Command(resume={
#         "action": "edit",
#         "edited_text": "The human-approved, corrected report."
#     }),
#     config=config
# )

# print(result["final_report"])

In [ ]:
result = graph.invoke(
    Command(resume={
        "action": "reject",
        "feedback": "Add practical examples and improve the conclusion."
    }),
    config=config
)

# The writer runs again, then human_review pauses again.
print(result["__interrupt__"])

In [ ]:
result = graph.invoke(
    Command(resume={
        "action": "reject",
        "feedback": "Add practical examples and improve the conclusion."
    }),
    config=config
)

# The writer runs again, then human_review pauses again.
print(result["__interrupt__"])

In [ ]:
result = graph.invoke(
    Command(resume={
        "action": "reject",
        "feedback": "Add practical examples and improve the conclusion."
    }),
    config=config
)

# The writer runs again, then human_review pauses again.
print(result["__interrupt__"])

In [ ]:
result

In [ ]:
result = graph.invoke(
    Command(resume={
        "action": "reject",
        "feedback": "Add practical examples and improve the conclusion."
    }),
    config=config
)

# The writer runs again, then human_review pauses again.
print(result)